# 🌊 AquaGuard AI — Day 08: Temporal Data Quality, Jitter & Gaps
- **Date:** 27 Aug 2026
- **Phase:** Phase 1: Ingestion & Governance
- **Objective:** Audit temporal sampling regularity, detect gaps and distribution drift, and establish frozen cleaning rules.
- **Inputs Consumed:** `data/raw/underwater_sensor_stream.csv`, `artifacts/day07/raw_summary.json`
- **Outputs Produced:** `artifacts/day08/sensor_quality_table.csv`, `artifacts/day08/rules.yaml`
- **Deliverable Satisfied:** Sensor quality audit table, sampling irregularity report, and frozen cleaning rules.

In [1]:
import os, sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").resolve().name == "Water Potability" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_raw_data, load_artifact, save_artifact
from src.preprocessing import audit_sensor_quality, detect_temporal_gaps_and_drift

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day08")

# Verify Day 07 completed
d7_summary = load_artifact("artifacts/day07/raw_summary.json", required_by_day="Day 07")
print("Prerequisite verified: Day 07 raw summary loaded.")

Prerequisite verified: Day 07 raw summary loaded.


## 1. Sensor-Wise Quality Audit
Assess missingness percentage, physical boundary violations, and IQR outlier counts for each sensor.

In [2]:
df_stream = load_raw_data(config["paths"]["raw_temporal_data"])
sensor_ranges = config["track_a"]["sensor_ranges"]

quality_table = audit_sensor_quality(
    df_stream,
    ranges=sensor_ranges,
    timestamp_col=config["track_a"]["timestamp_column"]
)
quality_table.to_csv("artifacts/day08/sensor_quality_table.csv", index=False)
print("Sensor quality table saved to artifacts/day08/sensor_quality_table.csv:")
quality_table

Sensor quality table saved to artifacts/day08/sensor_quality_table.csv:


## 2. Sampling Regularity, Gaps & Drift Analysis
Evaluate timestamp deltas, detect irregular sampling jitter, and run Kolmogorov-Smirnov distribution drift tests.

In [3]:
temporal_audit = detect_temporal_gaps_and_drift(
    df_stream,
    timestamp_col=config["track_a"]["timestamp_column"],
    expected_interval_seconds=config["track_a"]["expected_sampling_interval_seconds"]
)
print("Temporal Stream Sampling Audit:")
print(f"Total Records: {temporal_audit['total_records']}")
print(f"Mean Interval: {temporal_audit['mean_sampling_interval_seconds']} s (Expected: 3600 s)")
print(f"Sampling Jitter Ratio: {temporal_audit['sampling_jitter_ratio']}")
print(f"Strict Monotonicity: {temporal_audit['is_monotonic']}")
print("Distribution Drift Tests (First Half vs Second Half):", temporal_audit['drift_analysis'])

Temporal Stream Sampling Audit:
Total Records: 720
Mean Interval: 3600.0 s (Expected: 3600 s)
Sampling Jitter Ratio: 0.0
Strict Monotonicity: True
Distribution Drift Tests (First Half vs Second Half): {'temperature': {'ks_statistic': 0.0361, 'p_value': 0.9733, 'drift_flag': False}, 'dissolved_oxygen': {'ks_statistic': 0.0335, 'p_value': 0.9816, 'drift_flag': False}, 'ph': {'ks_statistic': 0.0456, 'p_value': 0.834, 'drift_flag': False}, 'conductivity': {'ks_statistic': 0.0417, 'p_value': 0.9141, 'drift_flag': False}, 'turbidity': {'ks_statistic': 0.0667, 'p_value': 0.4009, 'drift_flag': False}, 'pressure': {'ks_statistic': 0.0389, 'p_value': 0.9488, 'drift_flag': False}}


## 3. Formalize Frozen Cleaning & Resampling Rules
We save the formal rules to `artifacts/day08/rules.yaml` to govern Day 11 preprocessing.

In [4]:
rules = {
    "resample_rule": config["track_a"]["resampling"]["rule"],
    "interpolation_method": config["track_a"]["resampling"]["interpolation"],
    "max_ffill_consecutive": config["track_a"]["resampling"]["max_ffill_consecutive"],
    "domain_bounds": sensor_ranges,
    "timestamp_column": config["track_a"]["timestamp_column"],
    "sensor_columns": config["track_a"]["sensor_columns"],
}

save_artifact(rules, "artifacts/day08/rules.yaml")
print("Cleaning rules saved to artifacts/day08/rules.yaml")

Cleaning rules saved to artifacts/day08/rules.yaml


## 📌 Day 08 Execution Findings & Sign-Off
- **Key Finding:** Stream is strictly monotonic with ~3600s mean interval. Gaps are bounded.
- **Artifacts Saved:** `artifacts/day08/sensor_quality_table.csv`, `artifacts/day08/rules.yaml`.
- **Handoff:** Ready for Day 09 chronological train/val/test split definition.